# Train EForm Excel Table Extractor — Google Colab

Trước khi **Run All**:

1. Chọn `Runtime → Change runtime type → GPU`.
2. Đặt thư mục dataset tại `MyDrive/Data`.
Notebook đã nhúng sẵn lõi train, vì vậy trên Drive bạn chỉ cần thư mục `MyDrive/Data`. Notebook sao chép khoảng 17 MB Data sang ổ Colab để đọc Excel nhanh hơn; model và báo cáo cuối vẫn được lưu bền vững vào Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import shutil
import subprocess
import sys
import unicodedata

# ================= CẤU HÌNH CÓ THỂ CHỈNH BẰNG TAY =================
# Thông dụng nhất: EPOCHS, BATCH_SIZE, LEARNING_RATE và PATIENCE.
EPOCHS = 30
BATCH_SIZE = 16  # Tesla T4: bắt đầu bằng 16; giảm còn 8 nếu thiếu VRAM.
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7
MIN_DELTA = 1e-5
GRADIENT_CLIP_NORM = 2.0
BASE_CHANNELS = 32  # 16: nhẹ hơn; 32: mặc định; 48/64: mạnh nhưng tốn VRAM hơn.
CLASS_WEIGHTS = (0.25, 1.0, 1.15)  # outside, header, value
DEVICE = 'auto'  # auto, cuda, cuda:0 hoặc cpu
NUM_WORKERS = 0  # 0 ổn định với file Excel trên Colab; có thể thử 2 nếu cần.
MIN_COMPONENT_CELLS = 4
HEADER_FRACTION_THRESHOLD = 0.30
DEMO_SAMPLES = 8
SEED = 20260913
LIMIT_PER_SPLIT = 0  # 0 = dùng toàn bộ dataset.
SMOKE_TEST = False  # True = 1 epoch, tối đa 8 mẫu mỗi split.
# ===================================================================

DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_DATA = DRIVE_ROOT / 'Data'
LOCAL_ROOT = Path('/content/eform_ai')
LOCAL_TRAIN = LOCAL_ROOT / 'Train'
OUTPUT_DIR = DRIVE_ROOT / 'Train' / 'Output_Colab'
MODEL_FILE = OUTPUT_DIR / 'eform_excel_table_extractor_v1.eformmodel'

assert (DRIVE_DATA / 'dataset.jsonl').is_file(), 'Thiếu MyDrive/Data/dataset.jsonl'
assert (DRIVE_DATA / 'dataset_report.json').is_file(), 'Thiếu MyDrive/Data/dataset_report.json'
LOCAL_TRAIN.mkdir(parents=True, exist_ok=True)

# Đối chiếu mọi source_file trước khi train. So sánh NFC + casefold để xử lý
# trường hợp Google Drive chuẩn hóa tên Unicode khác với tên trong dataset.jsonl.
records = [
    json.loads(line)
    for line in (DRIVE_DATA / 'dataset.jsonl').read_text(encoding='utf-8').splitlines()
    if line.strip()
]
required_paths = sorted(
    {Path(str(record['source_file']).replace('\\', '/')) for record in records},
    key=lambda path: path.as_posix(),
)

def normalized_key(path):
    return unicodedata.normalize('NFC', path.as_posix()).casefold()

drive_index = {}
for drive_file in DRIVE_DATA.rglob('*'):
    if drive_file.is_file():
        relative = Path('Data') / drive_file.relative_to(DRIVE_DATA)
        drive_index.setdefault(normalized_key(relative), drive_file)

missing_on_drive = [path for path in required_paths if normalized_key(path) not in drive_index]
if missing_on_drive:
    preview = '\n'.join(f'  - {path.as_posix()}' for path in missing_on_drive[:20])
    raise FileNotFoundError(
        f'MyDrive/Data đang thiếu {len(missing_on_drive)} file được dataset.jsonl tham chiếu.\n'
        f'{preview}\nHãy tải lại các file còn thiếu vào MyDrive/Data/Raw rồi Run All.'
    )

# Chép đúng các workbook được dataset tham chiếu sang ổ Colab để đọc nhanh.
local_data = LOCAL_ROOT / 'Data'
local_data.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_DATA / 'dataset.jsonl', local_data / 'dataset.jsonl')
shutil.copy2(DRIVE_DATA / 'dataset_report.json', local_data / 'dataset_report.json')
copied = 0
for relative in required_paths:
    source = drive_index[normalized_key(relative)]
    destination = LOCAL_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)
        copied += 1

missing_local = [path for path in required_paths if not (LOCAL_ROOT / path).is_file()]
if missing_local:
    raise FileNotFoundError(f'Sao chép Data sang Colab thất bại với {len(missing_local)} file.')

print('Dataset records:', len(records))
print('Required Raw files:', len(required_paths))
print('Copied/updated:', copied)
print('Data ready:', LOCAL_ROOT / 'Data')
print('Model output:', MODEL_FILE)


In [ ]:
import base64
import gzip
import hashlib

CORE_GZIP_B64 = (
    'H4sIAAAAAAAC/919TY8jR3bgnb8inbNrZapJdrH6YySqWHZLXS01prsltFrjAWhuIotMVqWbxaSYZHfXlOu8B5/m7MsO5rDAnrzY'
    'BQxoDgusBv4f+if7PuLjRWQki1Xy2MYKM13MzHgvvl68eF/xYr6uLqIsm28323WRZVF5sarWmyhfLqtNvimrZd3p6Hfrs1W+rgv9'
    'PK1Wl/r3eV6fL8pT87i5WOjff1dXS/37It+c699VrX+t8+WsujBPpoJNcbGalwvzvF2W02pWzPJN3plju6fVYlFMqZW64bNinm8X'
    'm1k53XAZKF1syovCFIDnrnnbjey/v62WBcOsoJnQGw3yDbaaPmwuV+XyTL9/srzsdKq6Xyzfletq2a+Ljao+iV9+8+KLr189e/7l'
    '0+ev425Ub9YJokl0n/pnxQZ/z8p1kqbR/Sgu5tX6ogcjtFpUG6g9TlOBfIwYP3/yxa9OXj2NJ9Eoip+cncUdMbAKrNOxv/vbukio'
    'YDcC7NNi9Ga9LdImUH91ib+ivI5Wi40ZfwA5dx76yyWWWS79t/35dkkTkS+wwDMermpVLFeXHxZ6wBZVPsveV+u3p1X11i3SnxYL'
    '/kcXflmsz4rZF/DGK7ndlAsz3zCMGZDB9mKZLXBE16HCDl6gtrMiO622y1m+LotazSx1hEsjgenST+H3C2h2se7Sb5hjBfD97KKf'
    'bzeVLokvOp3Oy6+fnrzIXj15eYKTRJOaFR+gAdkmP10U8HuzzqdQXfZuEKvSz56/OFEQ8/jKYrjuE/wFEP0i7jz7+vXLJ2+yX5+8'
    '/vb5168MdvraG/QP4s4XL558+y1Bfgufk7jabupyVsDsx+cF9gF/vcsX2yJOO8+/fPX165Ps+aunJ7+B0r3BwUHnzclv3mRfPfn2'
    'q+zp85fwcvCw8+zkyZvvoNwXXz159erkBSIefBLdi5yinVffvfz85DUgy+j9a+zKuuhPq4sVEHuyjv92di/5q+G4353Ar/RjqL/T'
    'gcUS1UUxy4p3xfpycw5LK8HnYVQuN2nUO45ewZIcdiL4j3lEHz9TmZTe8qxd5Mttvsi8b+VcfZ5uZ3m/rLP8XV4ucA6SlHFaDFRE'
    'oMnyxaJRzWk+fVssZzWWBoqfFUBtF+WyrDflFLqLS6u99GmxnJ5f5Ou3UPJZvgAuyv1H7pjV+bxIaF6GyFSo6/B3qPtBn6KypvGI'
    'qjX8LJf1Jl9OFVg3SoDDdHHcuhEsrkUqurgugLcvGYnG2EQwh8W5aYFCCGTdMIpz6PBGAaVRAR0h3sbP7diTENttayRUUyFl55uE'
    'UaqPoiIevOmiyJfZBpaUP3pQdChBgRbr7SnSYX0PF0GkeLLpnhli2Pd4mKlvMXDhPhQsV4mudF4tZrvrxI8wzWK76i+xO4vyt8CO'
    'Xz17CpX7TU/7i+p9gXuBRBHH/b+rymUyPc+RaRRrZOORfSqXXBI6IGubwjCfVetLC5dGfwHYXi7j9KZRQXz9dbFa5DB78Z9+h+9n'
    'zVHIkKVm1PQEOXp9XhQbGotutK7eD5kUmTPb9WyImhjyKDKQxKETANQwTjtpPly6wvJdsUcoWqRdhCldtROqLs6KtT9Z8Dr6e8Ff'
    'glRLK6lBowjUDoNdvdvaA6FrZhaAaXcTGXy5cb3BZDE+mOL5drGAtTQ9h4ke9+5N/go4MMwq16RmdUctpog/I3qEqVT2tlzOxCDT'
    'Ws8zEJjgmVojV0iIpyU7eoGL8ob2xqeLfPk21uhN7dEICH8e31SDxp6vN/X7EgS1eBSHakG+tF3k8W2JJsbXsOTjn88jY5Qbq3W+'
    '2IWLlh/TlR3CvRu73F4U63Iad9xqP2xiPen4kKHUn+AvO8UL2A7HVO+E0b4rUNiBtT4+6B9Moo89wQGLGOY4g2INSYJYVHz06piE'
    'WM178Z80HQ8PHz2e6GHALlpczUVILaG3q3w2o9rm8X+5siDX/4l7fLbOL2psMpcbA2kXH4b0L0g+DybEhfkRODAJlMlF/iEZdKNF'
    'sUwYCoYjOkxTbh0CIFYsT9ht62blWVEjs1c6VB8I+W1xeJpgORD/kasn8XYz730Sp11VPKuhvaOHaZ8f1T5Jg7m9OC1wvIEC+iip'
    'ZqeXm6JOuCAw80W52SyK2ELwuIwV4H92J2gS3UPZsH9AA8xF/jIaMLeF14SlnuYLWGgsI9TfrzdyNKi7qcs+xrz67ytIHB3FD/Q0'
    'TRShwe6FQhmJ67W30xDBbbarRTEmcod/FNFB9RlsJtCkA/PM+4p5RRspqQVLVCLyDUgDBj0ME21wNdDc1XXKO0oteTE3d2RAeT9S'
    'wnWXmJod4aB8gYsyCa9KWkx6TAIsz+0ijrV6ojGgtvThKU0bEGYQNBC/EHBq+03NIF3QHpsRleNg2T1bfaGB6tN3QdaLYg7DuKlW'
    'IA+UZ+ckl242oDiNGhoYyq6JrEY0vKWbjMst1to3qt+hP4PHFlP0xustow6GBJvakWxqK9qw1P+mWNaV2uTiOH6DleXL6AQVwIhQ'
    'IUAV5Ty3PZT8oy8+fPXhb0DwQtB+h2BfMQO+j5vBfbXnQJ0gEyw3NVhiCujJal3U8Ah8rFouLqPTSzRPFP3oDUmENfRttSpmLE6e'
    'r6vt2Xnk6izIcEDnAnqroEQRLYsN9jiakvWHaqhm2ylLVdzguq+7xvRR5Gg3Ql7J/f9tsa7qJPGVRh46M2rIxrCtIwai7eLBoRFn'
    'sCxu2we4BhQIPTdYuq6/Y6gVacXw5AHXCzx7IFaP2txYbiE27ywTZnLOq7lph4NbNw3RO+WhD4nilcORFUGDMjM10Ui9sFkLHmGQ'
    'TmzjizMwy70DCi2+3wKxIg9CLut2Ko2OR9Eh7/teb0e0RSdOe7lHPehSL4SrF+EX0Uf7a4+RcdnWnoK/z2b3GDoHDqVRBPNl067P'
    'sI2ciHpOTFw3jl1cUElGm72uip+QkQGx0eCoIYDfLt0o8hwfdNuwoCmP6D+hJqOWxoJsGsY02BcTir0ksbUgOrwNIi0PtuB6cBtc'
    'WoJuwfXwdh1UknALske3QaYl9BZcj/fAhSgSSWH9eYW7BeBezIDCyPKTpi01/HKPGhxIklVlbSDGni0vCqryvFqXYNLewOgoeQSX'
    '6FU8xT2DLIH86wsoUy631baOrx3sLa38ZGcr6+1Fs43OsDgNPq3WZFpFM6VqJTSs3lwujBSVNtAh00EA7E8So6CBvaFdHn+AzBHT'
    'kKOI4M0mWtofKqm1ubZ207H5BDiUfEu7S89n/xbhbmJ2uInBqXlnA63aRqKRZ9Roq3wf4tc4fcFUvz+KDtrQ77MeGruVi0xZuRq2'
    'MNxAXSamrXEtbdm9ckDgSXBfI52xGx0eHCAZwB+PENBijMpsk+BMRZ8Md9bE8gxLcomnIwMh/weRqQ0aJSuTojbHoVFP8MmW9mvC'
    '0VQ0qkXrLr1kEc+XzGFQEXl0zGVxg8Waj1VBTzpgXmQFHzPyn3KDYVEMGZDbTC8IMQ7/QEyonbMDA2ph2ksP9qkIu6sGpEfF7yHy'
    'Q0VXQbyH++LlYerx1xDehuxLigtYrtAXRho/eZrqpEVdgYbQ9yEba9BVOtbeg8mkRZlh9L6cv0uqX1TLM6tFUpVkr+a6hQNmpUZi'
    'xN/GxMAnPF/qDfH4iSNf6UHk4TKwiutPDChvC1aAZkdYZshZFXNei+Lc7zE0aegU4TYL0pME79ZxFCR1hdgpOgzjPpTTznDaOrJa'
    'LS6zfHuGez656xNHLxs6M9kVMxn6si6mJZprLUWA1jtRZvJuR5hbHFgH02QorXEKY5va1tW90WI26KNECGiNYFj0kyex+QYb+4Hi'
    'ZIr3toHIzxLqHJRb2Ew8jRR3LNWofn2er5gDETAw0BY113QrcbHCqjUNlrqQbJJZLKZaeuxYwUtX3r7yblUPo+Fa0mDvxmpvo/fD'
    'roNmOBEjFGjeeCegplmWV5fFOmeTMPfoS/0mSR1Xrj+j+JJnUrUfpcRstq5W4Oc2IocDIUsgaF+TAdCn89GSKL++yOu3poHoe3aF'
    'WrfzTDFd27WR+WVVZmADTo0dH5dVNIafdkUzcAA161fcYlefVRGSPVp67mIRrBhhVM/d/opx2Lvr2GO3JjEYfxklPn2PpLzbGJTB'
    'L4dd20A5Jto57EF0vUkCjjld5HXNtrgv1+VMhXMk6q8yVJB3I0Mvd5bZQaiLxbxrrY1gFoPZWVcV8FEM6+kKDjcFlaa5tyInndhi'
    '5XK62IKdUbJuAEJFiQt54Q+6DX1RNcoL9sktpppBURj0y/0MLvwL3UZhQA8y/gkO9njiIpjm03Nnp7hxX0AsV9eO4YibpmVpfkIR'
    'oSBzA1g+E9V4z4Zku9BHEycQZeKiCuiN6CULjnlD2BfYgdwKf+kbnt9ofbkq0mBR1Vf4TkYyguPFKpuScRF0OYwnTURKe2DyBG0G'
    'qBNbqp3ZjT0WFR7bFQl9CtROn/SY+zv+bTZ6GrH8Aspm5cyQ2zg274QshUzXFCXPx1JSk9NlHSWG5k0ZNZaE50sui/u2EdUWgt4y'
    'jLeLxeKzIwWI0XY+IrtMswAZB3cVeFsUq2xRLt/WoRKesru+bBKbUfyUTRS72Dcv67HpCT4yrcWTSbMnnizjghGPjietai0AhN0e'
    'vtm+gcHKJmHtw4B2TZP4a+xROIQVQchVywDRoEwXFZBt2mQERDxjQ1hSUHHkS7lXhODk+gIYXDZqjXXZ1yt8PTesDZ+TTobe/uBy'
    'DTSbmZXKzuaJB6CLKIY4lmgmDW2zayeGoHjJM0y6q3RAn/BLNVndBRj0KGQSGLwzPYIJDEOMoeuVFqt1uM8adlbFMAquFQ9CUd+w'
    'QY4+5hW4ySVOevZLqYEqVP+IwVtDmi18LVmgrxPdRfex9pK76ED+xNlRaDjD8L+r4KZGWvrQVdmlPhIGIkV+6Kn1rsbSAqk086Gv'
    '6O9RKZsAhp5FYL9qXbvAsMVcsLsR10ELtrGI+KTolJ7cpDt3zTTqoEgIigfZKTsDEbdOTjHwy5H0bsG2WBJwXzoBFrWyILq683jA'
    'ETK2qRn8D7tKrUm9kIwWJId7ICmXKwhx9nVjFH24EHhcGp5o3XTp/N/pkeZxtrVgPJ1bSRtKGWLdZhyjzgZkcTLGWclbT3JYsbDl'
    'TGQSxIA1SYU2NY0qdWVt7ozYqTyZQip8Qo/BCRirGkFFGzLYUME1zAZiQA1UE8TbtnmMrLDvCCXNjUgXdHsrlcWrmBseD1UP0G3E'
    'raI1Tr/QEUUVwyv+AW80Oninf17rmLgzCLJYwfRvMZZGWYSs0GCEdI4Fg6I1ObDAnwZuokMx8rBJaHAIx1JF3fgH0Rf+Lns3MLou'
    '+Pbefb6opm8TCEF/STJySM3VQg6MROY0vBuB2t54G9JNtyuMVe0blKmrL55iI2BioRnfkitoU+YLV5aHT9jcQzT8yCobbehG4KfD'
    'GDuIXRmBjfi0zJX4nXZ9jF/i4LwCj3PizI6HMW3UEcB08uK7pPlaNbnRxP+QbRZaIBDh+3w9k1PvWYtbHAG+GE0zy1OGWqY2sjyD'
    '9r/BDe0VCCI+8UH40JMIz4ZAdF/0XQ+KQPxRDifKELacg/sqKkA652AROE6kzrHc5133PofFYQzSLUgZz1009gKUixvFgO/fnsBZ'
    'gVoPAN4uOp+QnerCCA4dBA6ABw+htIf+KqswtHNZ0FJrwYJgTUwPPEyz4qbWANCjm1uk8Ax24nmwe2Rw1pl1qMXmjQruxOLkEyyM'
    'wb8WoYtZdWbZ0LtX8NAreJg866NIsAK7HjRcg3dxCqZFucjw8BYfyksdz5aZRW9eg/gOd+MTc/msj4799apa0I5v0GIABsQTa3xK'
    'AusdDsGPRijj0xLMHEWOcSMUZAJMaQ2GXs3EQpU5hJSog1ZgrU70u64ZNoprvhgNmq0eNFptoWWbBz+/zQOvzYNAmwemzYNAmyVf'
    'RKo1UPrUzHaFAZ3QjuV8W6N2bX55/rgIgjtRxmOzsCeys3gSIGLLsoBFkjlOy64QVyaF0Y6lWVFqTFAYo+8vKdBvRINUOeWFUx3A'
    'OHF18g0MrbRjqQr609U24RgBwHu2OR810H+MnAOMGzSHiQhU8Fe2/0b5y3h0QSCDILWarVA3jrGn28gzCxDHvi4/UISOAu1vqkSo'
    'Bo8f6oOBEF7Fge8A0MfgpxQsVWDiyNCVrxwaql3DRm1oMYkpsjGfTrdwUusSxEp26SiMszI/A5SEGAybVF/K+uR8kNVA0EZ3MCib'
    'msAyvyhcYV+OoXAAAeMA3lKXGLNjuqXldN/YhIsoUHqoS6pW91y0HryOEGrUNrwJfoWWDqK8kdfw+1HivrjnNTYwR8rUllN06p7o'
    'dNtb0M2RoRzCarIt/VjXATjt23vqrYOn6A3EZgqn+wId9fvVaJlVKaptoF+ENdx4RbNjOI+M1HOdmebGNrzLvNsBx10TQKqv7RDz'
    'gSg939Um6JUoCk9y8NXq0MqgQWdiUgqUrpfgHwxT/QAIeaCpsAGjLaaicFtZyzmdRdKKW891s7p7LWh7rd2R/Gcci1Lu0LWOxv1G'
    'wxxySV38F/l0Xen5w76ZWUDmhbzbvvAgMQJQGdU4Zls3MDFl1Gf+cM/CivIUJ+Uc9OBCaoOwQ5ZtKu2n6Chmor6EAmPgHMRpjtLE'
    'pvRjapSTFA+uGZMvHrC42OIWBNHJS9B1OZxP6xcPu9KXP8cDutSic9z/IE5YbQHk4n5woOJuQlafodd0gLAPvOt2QpYcUchac+w8'
    'OyVQcmA/+ztY6RsWF6yxDZxfbwtBPq6Vi04XKjFBDYXerYLmLm/3cs5yYCfEVhU6eKAtQkPf84sWedvGsTg1MKFjVtwz533TE9WI'
    'TeRokZzE9LFzhME14YbQy3wBtgI1RjcaA41z7By8ItyGQHu36zXSHlerHswZKQLqr6pVkgZ6qhpiLG47cLXEZy8xVIOK0y87VS3e'
    'dFGDOlLh1tK9GezencAaY4Nx13cAuxcES4dBTHhAKPgB/zuIjkZm/CCIBim/tTAGa1sA1ZYjvdh3gokVEZytyU5oOpKtKPsW4C0D'
    'ItdJGFtoxTRWoiHYII5mSAhuSYba6dBUG+9uXyRGwrCIOspvEIyyDRjsDSQuEMEqO43KtNMFmYnmkcotYspMAmCW+Y8VjSBw1hU8'
    'NABPsble4LdsiIoY9156ceY6QNcFt04SF0Mj1EBtk7rfEzeCSO4QsrXNg2fedmtEn8TudcyhbewtYplwYBqrfRigCIJK0iQkHztQ'
    'Uvv+3qAl0UdNTHh8tzVmGYdMwHR11quRE7iPyitYTfmMoHOcsJy7Io30WjcC8i2OlhNIvNfT+kvGDt6xP7ZqLTsH9yaWjkPEqIe8'
    'JbKGF5keNS/+QLmoVXx+J+SJNlH2nbC32dKTV0I7lQ2leN99x7E7h83ydqShsH0QQQSp4Cp9ED3h6FpxOQJp/HSW8+shaKrCH++F'
    'z3dvipF3z+hzRUpyPq0+oISdzMt1vfHjxcASB3auZTiOjIhGBYWjFa8utACuj58QTtNkRqUe0yacOiogAHXvNCQ/B0AdTqaA7Vho'
    'cP0mgECyMgWvhs6C67FsQKtGH3SDjeo1Rwd5GFgMEMgQgQ+tD4s0h+ieOpKg1EtsbQbnxjFKx+876o9iDnTFXhdtKRO2oXRz7rrB'
    '7g8kADqTavB7QybK+TXYnDR2OM2BOdG3e05b3GExSaPQ7wS5DjkdGwT+1noLwMxjwbDdqPiwKtq/77AkKuXDVGBEJ4PRNx5fxWqt'
    'ARuAE0p0kJGes3yTPTowb4sPsK+AdeUMTTL0ks2CJssZWbPB3EJ7I6Ul0Y3QAfSUFKggW0CbEZFrIXOukznjDPYxsig2uqF6bJrh'
    'Z62wVWqmjbHpu0+BnWLGE33oGmfdYO9GggeqKD3Nq0x/VUhdl1ot9m6yRADGJoCtUEN1dvWAEImNFya5REO/1aRdnNZtJIb3Hsdv'
    'YeY5g2AMvSMJBNvADzj68IPPvqqjrvoIrFpx8jCsuw3Jo8dmDMHuflG9A3eQaWCqvGfLCjidOqshMrroSVf2eEFXToxFR+ymmqLR'
    'KCRHEA02op5uA8ZQPULyhB2jYeTRAR8DpjflMtoPqbdoWKBx+0AZEmQnack668AroNQ+HQay3i6zYlVBuq2OOitQLIaRcYFz4QWl'
    'lBzK9JJsJ1mDCrQmM5QHMSveldNCG5/4ib9UK0gUBa6wtf5IL/pf69fmABejqaGOFVu6kF3phEezksTvRbnKMCGSNUIdArv5s5m1'
    'wlwTMmRCXZBodWQ718iLQQPbp6KJBkg9n1QgMGy3P6n1DCN5XjKIQa7F0ROkUfqAvh2beNZz8YgvbIVWuyYHwrq8FcTnMxgvireC'
    'fKLJQqUexWkbibnDlufvCunJxOVAMVw6uJgeXLePRt911CMTRUcQYx0bNUGnlyI0GHoQVdB7jwEsnEXWC+ey8DqSal8EeO5aTaC7'
    'W5jJpwnMkEpBaNig8RbQFR4aTNumlREohKWzYokcbmYJxK1gUZ2V1HIipoZXn4vQbJiFmTCMccM2VMFwVzQqygtKAQlNuxkANxci'
    'nJQ+CNtMTO5dleMWIajTCJYlvD5A1AFXCUpB6IJtom+2wg56vSmkeVB4gOlUBg5DHwYBdwb2hFt3Vauf2/FsBwZRrLN7WuOkcYPU'
    'RRASJFVvV4U1LYLcoRvbJAMn7OF+nKQbTxiIihzrQMCJRz9m21DOBMd07vsYAoDkTW8Jg+wGgKRu3R5z2YRsYd3dNktEk3vvOtnC'
    'EjUkbV2Sy7xdvFbD0dWMwgRTTpSI1no8hIWers0SZ+vkYxNJwLBo2TOLTkBQNhGnd3jJyGADSV/ELImbgEdsMy8/MD2MwAMpWO31'
    '/SvcR5hXp9cgdiHFYqEQ/Q77D+fXJg9rDTsD8p8bQhdkcVCPAGHM+SbMarkvJTTVFLsxuINnB6Y5eKoOFjwjnadP4ZaD5WWSRSid'
    'Q41TXCsSWOWX2JxGMjVlsQiYDEIB5zJNAIrJ0s6RduUrqf77SQJEMaPmO8CO9q4OtdgTEWObrXmfXF5BJ5U0iqJ+ewuDZsceCKDM'
    'HrVrRb5jUhH6mHGz+DfVyz9V8g9+uFOyEdppCCP9EuYbUQ3aLdkSLNtBSUlcRHpC6UGaY/QH2VKdl0T2QeQC1KHWqoGw0YoqKPWc'
    'afCxqPCGjCU8My0GSZhetO7pGil3SMMOuNBFVH4QGpIGnnqVL3VBY0MymEP2xYUE0YYjW5EDct2myvFxrNc4yahAxVeNBPw0hen1'
    'FTTjehj4zDNwfcWNBm7Z8Wymr6FzFJhvVqV32iYVSVcssIrKxz/iLU3HF5TKE/QdMJqCMMhT5JoOgA2C9RT/jGmWJnxYYkzzMUmF'
    'a8+11FIL9Wkr895TCekEK2QWLufAFlkwy/COCXV4PKjbiYOGTjB1y9lCUH6nb1cVq2JaeQGuayvD9QVTXU3pXN9Iy+XvC5yPmo+5'
    'WoGaziBolNw/TsmeQa5+CgOilOLunQRiF8lLyNf0a2SPJ+s1pHaYx98tIYZ6RTMQ6bEYRldeLR+5tXyU6q2SRhc1QgsAAgT1jj+p'
    'g1P0Coq5EehcRKk2JnYYZ1l/cYKKYcat7pIKdZNmElwdINySPicbY1/rDYRhCtgl3ET29L4regJ08tc8aeVyXqyRhiiUF8BE3Epm'
    'OPou04K3z95gPPi3jVXxI1FasvveIROo3gdve6S5v13WcB6lgMz8B40599RDjTwdH0w6e2ogB1LxcEJ1mgqUU1b1a7c6Yb92w2qR'
    'yFlxkwpwg+ifujFUfKdJgJfxXSdN7pZhyB6ZnNyQqVaJb43nhGrUGhgBuhYEKsrUifevxOZqAn2FgiB5wwz5MRFcHK78iPk8rl/P'
    'SCEmy1/rHSLWiEup6WDmbkDnAvjlhLOmwR50EoTAFqKHV/ubNPcIsEvm4n4RznDd2Zlzwc5pt5E0oZEkoZEUQZ9zLNhIpFaqkw4B'
    'xVebCkHJrl4uBP/w25kyRLQwR0f9pQFtT74QyPwgWKSXgzHxR5DHtWV9wfA+TAOBQTr9YghV60qkRD4PDnx8XlIEGq2wAGpPzb+C'
    'JQTii9URIOHBItBhe2J+fINa11Vzkqp8J/ibjzfTTHlWiWvtl/TyPgTzPTTFUL4SaRgQBpA3yIP6/PoZSRUegCfQSCAWc59xKgJL'
    '/n3ELovRWNP5UPzhSn2wsC+qTGVsSHYnBopIpVZnLt07kHbspmQhVzcjvaY/4tIiVslvCj8Cm3K5pqjOMbiTyAOEZEaZ/PFEUcKG'
    'kft4lUE3UkWqCsuoL/jJKqGUKKHLf5RRBWhA1TJsusg4mIlzXNicQPZYfK1SJsgsDMhS6ae64cQ9Pj/xTBk6iw+MVp/tF9a/VrOG'
    'KFrL9hNbIPVuLjCGjl3y2um2XMxo+utdklpLGqk9Uki1y3OCjpRvXtHSv4e4RyPgxfSa+Q0uD0VVvAbSJltobE37JP65854VTNvz'
    'r5Cyx4YD/BtvX39eO7D2iOKMk3ZESzLdne1icjdTso3JcNtAJLdr87ttkhmRaGa57565V/YY5ZDmTiifdE27Db8KlLaGdFvcjkiz'
    'vB1NiinTD7vawWfK99ruQ7Or8YT6qpS9n1mD6W9YoGhNJrVLsCCi0RcSgls1g9jud6BnnQNng2vU2vZsq+R4py/J/U+Mj21I9Byr'
    'a4PgDfZD4VacsTwDxRKOqX5gHxLsWnDrEV7JWaPZ/QEGOZ3RcdcE03o/VHsSFgddtI8FE660q+skNyQ4VrSzIFQ3bHb5abEYqcI3'
    'I6WDn2Q/2hezgHDRkx8X108Sv0A84utgn8rFWaOdDaBy0bPBrStoHJbaPYJEn8+r77x6RDd/bdBzYgPl85EAh/s0TEfT3NAiKOa3'
    'Zy/0TozMDXVQWSXhexWFO67UArfrWEX+oaR0KAhuFy6+JUwfqMIkPqGVlLoFMNMRxE2tzvPRQf/wkfd1AYnsl9rdzgsNGDfYPLNF'
    'fgnMyP2Cyx9+Jkq3XpWjwSMVnoarkpkHlzVXIdZwfZi63jUJerVshiAtaGFScpEryLw+dO/JUx4oYWQmK7fNEOlGxxuTudQjyZxu'
    'k3+dVrPLxjErk4+O3U/W/U3pwYa69dL9TYKDG5G/yVEfiTfnsbLdqZi9I6eZbC7ZzGJH+ze+SSjDFWpvdtPfUtBY4R3TfYiIwRPj'
    'cdy8UtC7mzS1l1r+7RIj445O18fe7RDsINMHRuOjK+jR9fGVqvP66D6/iGWuhtmlBoiPNoARvBLm1k5Eh2fn46P7G1OZCpBEyObF'
    'e0ez8h2nQRl9BHebbC4/Ov7V+Y//GxS92U9//J/Rn35X/fj7Jfz5l3/66Y9/mEanP/3wB1Brju4D2LF7W98REYLTHqxSN4c/yj2P'
    'xHEc1IS2w1tvenVhYkMEcSEu1gARZycgKuKXcUMAmcAeQfc4hr/y7Ip8sixcGHSurOEi8741UOmO6Hl1/WhH6vOxdxdFfHT+4PhK'
    'EiXV9hHJfh9N0uvo//5zFPpuRU8sdXQf0DRQr45PTASygwPpm/H4kiQgA2xI4x6yJ9T9VjwNGbMV0Uud1qAFkxU4GcX9lYfCofZV'
    'Xq4/OsY3x0fnD49/9dMP/2cTfb8FAgf9B+j/T78Dii9heB56WO65DDisFhDRE+5QBZtzWFzTaAOvbq6AyacNvfq3SSTKKl1NtxfM'
    'vyB/0dFfwDNdUoojeHyE8TB0uy9seaOP6MpHGBJKBX5smPcVXinUm+cXJYi4T9aQR6tb58u6V0N42vwz4JqQwOYXg18eHjx4cK2a'
    'ccXX7PTwFoPD1Qfwg2J2kF98+jB/cPrJZyot1XCAn1QUeR+n42pWAvXml0PcYD/Df3p44RJmYenpjWswX0fw/8/O8tVw8Hj14TOI'
    'JF6DkeD9EG3e1zRkunrKTLiqi6H+8Rl1BeXa4WCw+nCt7qrrbmYKZDiwrX388JcPPzk1rX0IdYE623tfzoAhPXqEj/kH9Th4fADo'
    'NudXGG3HQtzwF7PTAiI34E515KtXaqBOPx1MB1OueXUl2gOD8Rmcr4WugtscbC18S+j1X1/A8sgTW9WnB1BVesUD1jpG10D+Yh6P'
    'zgfH5KAcRm/DhB69+/G/eXQ5OAaSEexc8yklnaDF7j1GKfJNOprSVMIajLrUl4hquWUBcX6bfQ2UXBiSYGXE0faxT+IBSxcKD6o2'
    'fXcyRTllPEM214inDdXjh9dimTRga9J9MzWrasaehj4xx+9ErlztYRRWSu4Mpq/miAKFLvWq1bVQ8cl46I3GRF9ySupXxk7GXZZB'
    'lfBtVq7lW5bmtQh5qKK1OT6NaFl9+USFnkNGJIxPzVDAswY9SDjyQDmQKSIA5IEpLH35XVkHgdJKtHhqvI+MaRFgFpiiUsA8ukWA'
    'eUs2NmXYxG0iU9EKQxUZQShUqLWIuE5QDejyQZlBf/AolVZT4YjUnjn+DEIvWeHAHaAr//NEvjt2WZqyw8cHnw4edI3RNGNDtanh'
    'sZq6wBrkVrYnam27LwCzLVWLd9oSYkkLg+7Ng1dqxlclkC/ERQfG1xgPM8T4QxXrY7icuqVaQ4IQDktmTwSqNOGJpfAs24FuWZSe'
    'Erq8UX4VddlCfowK+nheVZtnuEdwqEr85rwEjrulwxn3nb5E59VPP/zz1PkiG9mPvvrxD5ewhf/0w+8vo8/RDaBul+iXq8vlKax0'
    'lNz/cdqP08CwwIgQFrSD1UmoG2TJJv7e4OppaIBU2ziRNMBeZsChMh2iHqc7InZCkwA9+5d/ynGL+uH3m6iJcITZY7QhACQmew2F'
    'DaIUPcT0bG5INr4hdUFO8I5O92khIFSduHeOwSsdFyDiJm2DvN1PtLax1ynF+VLtYTLNuVqHd/ZgudGfZltJlEWOb5TWxhlMUms8'
    'dNYTl3ZskmmoKl9e0rXSur1q93Er0B93EcCV6hw6xPgSCenhozcSlQ4lvjaEqMygR3BduDmbghuSfSPzQKqXggvj7YO7CFSbsSxm'
    'PyHlChYiyFJwYmxAApVEbr8dWM2cnH9yp6EoN/+U0M44Nwdc1zL98X/BqvnxB5Dvrnxs19FZ+ePvgTX89Md/sC1p2XPMMMEMwzEk'
    'FKha9x38CHvgriFsq8UfuPY6VMnNGmVXvOh2MIlVAgzc47IC9IHLDd4z7fimG7E9YnP+/y2ehywP+K5Puh4i4Nbqw8CtbfKn7jX4'
    'RssLPXlPT379/IuT6PLH/7GNpj/98N+30RffPX0SLYFHw1x8c/kG0UZv2X60Ov8RWDbsbH/8r8voy2++o7JxewOh+/KSPn2qClcQ'
    'Tj6yb3UoAD3on+hM8mdeGeM/X63ROhk/Zc917MUttYzQUES1ETg0HGHFgGGFgnaSA7c+NQoWKNPxHlnmFPxWSVwYD3VnxmdPL2Zq'
    'B8NIVP/CKSH1dC2b1h6bbviiIhGba7eEO9Qijf1tVQkXuNpw7tIdtVXtXQvsZXetBrfBferZ//I5y6v4LE1WrbR1Ux5sNjsPpgqx'
    '25AtIXYcKCKenIByvrFhzm5acX2DKLRCDRFUgzXmCA0uFlkYSbxGo3W4cjxU2BViA5Ms9xR6aE8lJw4x451q2/l8wScvg7e+dSGX'
    'qztiDZoNVdMkaVuXCtEI49X0GWy7S7u3wYi02IpSEOoeOHdEpTvCysjNge3HIkuxIuMbhv0Lhx3JI3ihhea6IwezPtqqkmzDUen6'
    'ZAkb+uoSPbMJIxwFGgAr7gx0+YL9PCOZYTh1j6abpvLZ9Cez/OJvgidUF+uRY6DoOuaIkXxQcaV47B8Yg1/FAoR3/QkSh9egDzyB'
    'kQU6W569eJ2YlnWjN3Cl4IeR2sxYptR7iG6aOaWNWQjM2z66ySC0n3YH85ayETR6JveaefyN+QLWc6+WYVcfedClP6eD3GzKubJc'
    '5vozZjlFfZ9aPWTRUq7lFMpo89AV/9DI6SnM1ZrGGuAezZcyxrIl+HV4c8TRjmjX4U1hSNe8uHaGaVh3FOW3oEMaFFu0ugQhsljh'
    'D0WJ4lyHdpYSiEoT0tMXRtNbGj9zbh9AYW8ycR82UQoXM0f6DIV5GcqUBZA9KhjU6SaR2BEJJqfb/WLWdbdzU2yYXQ2ej4f87Yp0'
    'hgeHMzjqqsgoUhqqA/Dxxw5ZdTvNQDHB6e/S3cY+cuc+u1d03dhdqPj2nbUcyDtLr2mqORqtuXDtQQ0m95YjhhxrNGQya+Q8E5wV'
    'ytij/sSD+LoRjM6B2J518+40mvChS6teGSFdDgOdaxwwtGxu/5n4+8j3ROo1AH7EK6d144/w5UcTOnMNgO/UaenRVWDgvbJ+JQgL'
    'adrDoE4Sd7c+HagUBtRfd9fLEXJABmEkbqiSW72KFgoDqo8KIg5QMN7ZyVl3JDO8J6z9TnMdhkl/m58156S/AehbcWiR/rPJgLVy'
    'Pmwv6pz4NzdDiwJgAjEOj0BGBt6hT/L14hLgqtUK09aAw+73pWL9TLa4V0Pn+N3oyo7DdT9upuA4Bd35bWfHsUM7UA3peidTVSeL'
    'mkxUME7NLJlBRvHn2G6WXx1bpMf+hL1Dy937NMWV33e14w2U7D1/enPNKJ7vXbUV9G+s++uvd1cuvCYXb+FfFBQxjatSlgoIRoP6'
    '3gpF3h7jcpwu4AF5+fXTkxfZs+cvTtBG2DzhK6Q17xTL0DuT65+E0VHMXANil5ooEB6asADfdjPl09QFWpz6y+p9gj9+CwMCCWem'
    'KViqKq5a3jAVkwguGoPhIE3DS6NR6iza0L9B1DuvO2ze4eTl7XRO8Q5dc7B/Ih4VGjrahHKjmxjK7lOiqer8p2hsvKtlaKKH3MPn'
    'eX0OswoGiVqFbL85+c2b7Ksn336VPX3+8to7wl7zhb2Hjx6rw+zWBTP2C8gTS9oF0zqUzM8yY6fUMoI/LKocCuxwYp/KoVqhpI7U'
    'H3DDyXC0zUNjWm6wkwTFE1cRdMtKRRBj8sWjV1Jzb4yVVz+9EmYvY32Ff3tlbq0TWRLTGrkmNNe/4EEw4zEmnsw9cLavSYlPIRQ0'
    'efjHH2jXv0UNc96E6b9xdLRBYz9DD/y5uqDfWB1i3Gjj3iIqLytlxRz6u1qoHJohh41NKNw+kS0AmgipKIYcv+olrApl7zGbv7E8'
    'O7Y8ttVDSKfITtAVe40ySf0iel3gzgdRRhAnC8tsUfTQRW6SM0BFKFiqIzUReMXr6B0GmeFhdwLCY9gIovDRHX55NN8il4ww9Ar0'
    'E2U/hexsEPIObtACrhXpKycQfZplaj/O9j7mrM+VNc7ZBdDKXd/Yj23YqHD7dtq0RRmR0RVhohC+6qQg+flH7Ln9KhTg6jY7N5dQ'
    'x5ooHY+d7+5dhAXDiDigkwb/5+zznpWLuLHzJrAP0ntawnf0yMjUn/M5xstBoDNriGo+1e7mmLnTnduqPEHsf5UbstowkXPxryBz'
    'cg4wq7cSC5/uzYinEKHT1gB/RRm9XjIm/UwtUuRFJHoK5uCG3zgip+1MI+qGj0PdAMWF+isIK+m4R/nCgCZg+R24PCmmto9RsLHf'
    'ThnUSLEjs+3FCsMVsQRGN9Yok+X1tCy1ER7N0cvN6NAmxjeJu9JQPCTngWuc++rKjotSXqR81++qtN6KFl/derXeKFztv5nts5Ht'
    't4ntSW67Z8Y/VI1zqROMYPgz7AFndUJRbfCLXvUxZQHH4mrjfE1Wf1PgyfqMQl6/oS+JSJc6it8gmbL7hV2L6kQh7LZAhdU6TgXS'
    'PsQYYwsIWxL3erx39HDvQEcy+lW+Udkwvt/iyXah1LWgYOLvAfHfFQML5hqaNnVN3ocHOyFJ4qa45iD0JzuBtRTeI6lcwauIS40B'
    'g0d3ImFpt8fieiuOhztxGEE+1IdHu+cPAsVZwm+tezcCLen3UNLvkfgfxnTYv2kqaggM11pjqCf62vkWBKQ99LRK4TZiiatm9MDi'
    'agTC7kSttnwLrvLr7IIB9aOnlZFQbw5umheUmHpGYurppClNTLuJg0WrnhatejJnSmieKI/KLoSkPYXXGkfr3jCUF1VPixghLI93'
    'rzlUx3qgjvVYQdsxsjpPFCOSzBP4aYk3Y6PYmGUUJ5CBNRrknEwF1ugq6cyxhBQuKRuU7gnRIwPdD4rWdrcXJe1LW5D5mijkmyWs'
    'wUAUClkRHIOBKNtiSHA8y7Z02JaguY8o2DQnGOuBHJymRaFpNxDldxkV3IABORhBM5djaRhRyHxigdrsEMLyIKrwlSJhdxClgtaI'
    'Fr3IHaM7ak4CyR4GAlzUAsI1jUg1z+l4SPvzDCaS2EKmlLTz/wDXT/KaabUAAA=='
)
core_bytes = gzip.decompress(base64.b64decode(CORE_GZIP_B64))
assert hashlib.sha256(core_bytes).hexdigest() == 'ca88e7e7ca098df71e461b9f3bf15de64244e2cc3da686283f2cfd7723ce16d8', 'Ma training nhung bi sai checksum'
core_path = LOCAL_TRAIN / 'training_core.py'
core_path.write_bytes(core_bytes)
print('Embedded training core verified:', core_path)


## 1. Kiểm tra PyTorch và GPU

In [ ]:
required = ['openpyxl==3.1.5', 'matplotlib>=3.9,<4', 'tqdm>=4.66,<5']
try:
    import openpyxl, matplotlib, tqdm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *required])

import torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CẢNH BÁO: chưa bật GPU; notebook vẫn chạy nhưng chậm hơn.')

## 2. Train theo batch/epoch, validation và early stopping

In [ ]:
sys.path.insert(0, str(LOCAL_TRAIN)) if str(LOCAL_TRAIN) not in sys.path else None
from training_core import train_model

report = train_model(
    module_root=LOCAL_ROOT,
    output_dir=OUTPUT_DIR,
    epochs=1 if SMOKE_TEST else EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    gradient_clip_norm=GRADIENT_CLIP_NORM,
    base_channels=BASE_CHANNELS,
    class_weights=CLASS_WEIGHTS,
    device_name=DEVICE,
    num_workers=NUM_WORKERS,
    minimum_component_cells=MIN_COMPONENT_CELLS,
    header_fraction_threshold=HEADER_FRACTION_THRESHOLD,
    seed=SEED,
    demo_samples=DEMO_SAMPLES,
    limit_per_split=8 if SMOKE_TEST else LIMIT_PER_SPLIT,
)
assert MODEL_FILE.is_file()
print('TRAIN_SUCCESS:', MODEL_FILE)

## 3. Metric và đường cong thực tế

Báo cáo tách riêng validation, Test-ID và Test-OOD; không dùng test để chọn epoch.

In [ ]:
from IPython.display import HTML, Image, display

summary = {
    'model_file': report['model_file'],
    'device': report['device'],
    'parameter_count': report['parameter_count'],
    'effective_train_samples': report['effective_train_samples'],
    'training_config': report['training_config'],
    'validation': report['metrics']['validation'],
    'test_id': report['metrics']['test_id'],
    'test_ood': report['metrics']['test_ood'],
    'artifact_reload_verified': report['artifact_reload_verified'],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(Image(filename=str(OUTPUT_DIR / 'training_curves.png')))
display(HTML((OUTPUT_DIR / 'expected_vs_actual.html').read_text(encoding='utf-8')))

## 4. Nạp artifact và dự đoán file Raw thực tế

Đây là giao diện backend sau này sử dụng: đưa file model và Excel vào `predict_excel()`, nhận JSON có header/value/merge.

In [ ]:
from training_core import predict_excel

demo_source = LOCAL_ROOT / report['random_test_demos'][0]['source_file']
actual_result = predict_excel(MODEL_FILE, demo_source)
print('INPUT:', demo_source.name)
print('MODEL:', actual_result['model'])
print('SHEETS:', len(actual_result['sheets']))
print('TABLES:', sum(len(sheet['tables']) for sheet in actual_result['sheets']))
if actual_result['sheets'] and actual_result['sheets'][0]['tables']:
    first_table = actual_result['sheets'][0]['tables'][0]
    print('ACTUAL RANGE:', first_table['sourceRange'])
    print('HEADER ROWS:', first_table['headerRows'])
    print('FIRST ROW:', first_table['data'][0])
actual_result

## Hoàn tất

Model và báo cáo được lưu ở `MyDrive/Train/Output_Colab`. Tắt runtime Colab không làm mất kết quả này.